# EXP-03 — Entity-Span Protected / Confidence-Gated Retraction

## 1 — Experiment definition

| | EXP-01 | EXP-02 | EXP-03 |
|---|---|---|---|
| **Name** | Right-Most Town / All-Match Country Retraction Baseline | Line-1 Protected / Confidence-Gated Retraction | Entity-Span Protected / Confidence-Gated Retraction |
| **Town** | one right-most verified occurrence removed | same, if probability **> 0.80**, outside line 1 | same, if probability **> 0.80**, outside a **verified entity span** |
| **Country** | all verified occurrences removed | same, if probability **> 0.80**, outside line 1 | same, if probability **> 0.80**, outside a **verified entity span** |
| **Protection** | none | all of `address_line_1` (positional) | exact Gemini-identified entity spans, re-verified in Python, in any of lines 1–4, when `entity_confidence > 0.80` (semantic) |
| **Town/Country extraction** | — | identical to EXP-01 (cache replay) | identical to EXP-01/EXP-02 (cache replay) |

EXP-03 adds a **separate** entity-identification prompt and result. It does not touch the
Town/Country prompt, predictions, probabilities, verification, scoring, cross-entropy or HITL — so
section 3 checks that every one of those fields is identical to EXP-02 before any retraction is
compared. EXP-03 **may** edit `address_line_1` where the text is not part of a verified entity span;
its success criterion is **verified entity span changed == 0**, not `line1_changed == 0`.

> Policy behaviour only. Nothing here establishes semantic correctness without labelled retraction
> ground truth; no model is called and nothing is re-run.

## 2 — Load the three experiments

Driven by the `EXPERIMENTS` registry; EXP-04 is one more entry. Paths come from each config via
`load_config`. Missing artifacts are reported, never fatal.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pytest.ini").is_file() and (candidate / "models").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the repository root from {start}.")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.swft_tc.src.settings import MODEL_ROOT, load_config

GROUP = "1"
OUTPUT_ROOT = MODEL_ROOT / "outputs"
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 250)

EXPERIMENTS = {
    "EXP-01": {"name": "Right-Most Town / All-Match Country Retraction Baseline",
               "config": MODEL_ROOT / "config" / "config_exp01_rightmost_town_allmatch_country.yaml"},
    "EXP-02": {"name": "Line-1 Protected / Confidence-Gated Retraction",
               "config": MODEL_ROOT / "config" / "config_exp02_line1_protected_confidence_gated.yaml"},
    "EXP-03": {"name": "Entity-Span Protected / Confidence-Gated Retraction",
               "config": MODEL_ROOT / "config" / "config_exp03_entity_span_protected_confidence_gated.yaml"},
}
BASELINE_ID, PREVIOUS_ID, CHALLENGER_ID = "EXP-01", "EXP-02", "EXP-03"
ORDER = [BASELINE_ID, PREVIOUS_ID, CHALLENGER_ID]
TAG = {"EXP-01": "EXP01", "EXP-02": "EXP02", "EXP-03": "EXP03"}

for exp_id, spec in EXPERIMENTS.items():
    cfg = load_config(spec["config"], base_dir=MODEL_ROOT)
    spec.update(cfg=cfg, output=cfg.path(cfg.processing.output_path), detailed=cfg.path(cfg.processing.detailed_json_path),
                metrics=cfg.path(cfg.processing.metrics_path), policy=cfg.retraction, entity=cfg.entity_detection)

config = EXPERIMENTS[CHALLENGER_ID]["cfg"]
RECORD_ID = config.project.record_id_column
SOURCE_FIELDS = ["address_line_1", "address_line_2", "address_line_3", "address_line_4"]
col = lambda key: config.output.column_name(key, GROUP)
C = {k: col(k) for k in ("combined_address_cleaned", "predicted_town", "predicted_country", "predicted_country_name",
                         "predicted_town_probability", "predicted_country_probability", "predicted_town_exists",
                         "predicted_country_exists", "composite_weighted_score", "combined_address_retracted",
                         "combined_address_retracted_comments", "hitl_flag", "hitl_state", "hitl_state_reason",
                         "entity_name", "entity_type", "entity_rationale")}
INTEGRITY_FIELDS = [*SOURCE_FIELDS, *(C[k] for k in ("combined_address_cleaned", "predicted_town", "predicted_country",
                    "predicted_town_probability", "predicted_country_probability", "predicted_town_exists",
                    "predicted_country_exists", "composite_weighted_score", "hitl_state"))]


def _rel(path) -> str:
    try: return str(Path(path).relative_to(REPO_ROOT))
    except ValueError: return str(path)


print("repository root :", REPO_ROOT)
for exp_id, spec in EXPERIMENTS.items():
    pol, ent = spec["policy"], spec["entity"]
    print(f"{exp_id}: {spec['name']}")
    print(f"   retraction policy {pol.policy_name!r} protected_positions={list(pol.protected_source_positions)} "
          f"gates={pol.town_probability_threshold}/{pol.country_probability_threshold} | entity_detection={ent.enabled}"
          + (f" threshold>{ent.protection_confidence_threshold} prompt={ent.prompt_version}" if ent.enabled else ""))
    print(f"   output {_rel(spec['output'])}")

In [ ]:
NOT_READY = "Experiment output is not available yet. Complete the batch run (see README → Retraction Experiments) and rerun this cell."


def as_bool(s): return s.astype(str).str.strip().str.lower().eq("true")
def as_num(s): return pd.to_numeric(s, errors="coerce")
def pct(n, total): return f"{n} ({100.0 * n / total:.1f}%)" if total else "0"


def show(frame: pd.DataFrame, n: int = 10) -> None:
    print(f"{len(frame)} row(s){' — showing ' + str(n) if len(frame) > n else ''}")
    if len(frame): display(frame.head(n))


def load_experiment(exp_id: str) -> dict | None:
    spec = EXPERIMENTS[exp_id]
    missing = [k for k in ("output", "detailed") if not spec[k].exists()]
    if missing:
        print(f"{exp_id}: missing {missing} -> {NOT_READY}"); return None
    frame = pd.read_csv(spec["output"], dtype=str, keep_default_na=False, encoding="utf-8-sig")
    with spec["detailed"].open(encoding="utf-8") as fh:
        detail = [json.loads(line) for line in fh if line.strip()]
    metrics = json.loads(spec["metrics"].read_text(encoding="utf-8")) if spec["metrics"].exists() else {}
    rows = []
    for rec in detail:
        g = rec.get("groups", {}).get(GROUP)
        if g is None: continue
        r = g.get("retraction", {}); e = g.get("entity_detection", {}) or {}
        before, after = r.get("actual_column_before_retraction", {}), r.get("actual_column_after_retraction", {})
        rows.append({
            RECORD_ID: str(rec["record_id"]),
            "retracted_entities": ",".join(r.get("retracted_entities", []) or []),
            "town_skip_reason": r.get("town_skip_reason"), "country_skip_reason": r.get("country_skip_reason"),
            "protected_entity_spans": r.get("protected_entity_spans", []) or [],
            "town_protected_by_entity": int(r.get("town_occurrences_protected_by_entity", 0) or 0),
            "country_protected_by_entity": int(r.get("country_occurrences_protected_by_entity", 0) or 0),
            **{f"line{i}_changed": before.get(f, "") != after.get(f, "") for i, f in enumerate(SOURCE_FIELDS, 1)},
            "span_altered": any(s["text"] not in after.get(s["source_field"], "") for s in (r.get("protected_entity_spans") or [])),
            "entity_confidence": e.get("entity_confidence"), "entity_status": e.get("status"),
            "verified_spans": e.get("verified_spans", []) or [], "rejected_spans": e.get("rejected_spans", []) or [],
            "protection_applies": e.get("protection_applies"),
        })
    audit = pd.DataFrame(rows)
    eff = metrics.get("efficiency", {})
    print(f"{exp_id}: {len(frame)} rows x {len(frame.columns)} cols, {len(detail)} detailed records, run "
          f"{metrics.get('run', {}).get('started_at_utc', '?')}, Town/Country backend calls {eff.get('backend_calls', '?')}, "
          f"entity calls {eff.get('entity_backend_calls', 'n/a')}")
    return {"frame": frame, "detail": detail, "metrics": metrics, "audit": audit}


RUNS = {exp_id: load_experiment(exp_id) for exp_id in EXPERIMENTS}


def ready(*ids) -> bool:
    ids = ids or tuple(ORDER)
    ok = all(RUNS.get(i) is not None for i in ids)
    if not ok: print(NOT_READY)
    return ok

## 3 — Comparison integrity

EXP-03 must be identical to EXP-02 (and EXP-02 to EXP-01) on every extraction-side field, including
the composite score and HITL state. Any difference sets `experiment_contamination_flag = True`, is
shown prominently, and voids the retraction comparison.

In [ ]:
def validate_experiment_identity(a: pd.DataFrame, b: pd.DataFrame, fields) -> pd.DataFrame:
    a = a.set_index(RECORD_ID)[fields]; b = b.set_index(RECORD_ID)[fields]
    rows = []
    for rid in sorted(set(a.index) | set(b.index)):
        if rid not in a.index or rid not in b.index:
            rows.append({RECORD_ID: rid, "experiment_contamination_flag": True, "differing_fields": "record missing from one experiment"}); continue
        diff = [f for f in fields if str(a.at[rid, f]) != str(b.at[rid, f])]
        rows.append({RECORD_ID: rid, "experiment_contamination_flag": bool(diff), "differing_fields": ", ".join(diff)})
    return pd.DataFrame(rows)


CONTAMINATED = False
if ready():
    for left, right in ((PREVIOUS_ID, CHALLENGER_ID), (BASELINE_ID, PREVIOUS_ID)):
        ident = validate_experiment_identity(RUNS[left]["frame"], RUNS[right]["frame"], INTEGRITY_FIELDS)
        bad = ident.loc[ident["experiment_contamination_flag"]]
        print(f"{left} vs {right}: {len(ident)} records compared, {len(bad)} contaminated")
        if len(bad):
            CONTAMINATED = True
            print(f"!!! EXPERIMENT CONTAMINATION between {left} and {right} — extraction-side fields differ. "
                  "The retraction comparison below is NOT controlled. !!!")
            show(bad, n=50)
    if not CONTAMINATED:
        print("experiment_contamination_flag is False for every record and pair: the comparison is controlled.")

## 4 — Entity analysis (EXP-03)

Coverage, taxonomy, BANK count, confidence, verified-span rate, rejections, and where spans live.

In [ ]:
if ready(CHALLENGER_ID):
    f3, a3 = RUNS[CHALLENGER_ID]["frame"], RUNS[CHALLENGER_ID]["audit"]
    n = len(f3)
    etype = f3[C["entity_type"]]
    conf = as_num(a3["entity_confidence"])
    thr = EXPERIMENTS[CHALLENGER_ID]["entity"].protection_confidence_threshold
    n_verified = a3["verified_spans"].map(len); n_rejected = a3["rejected_spans"].map(len)
    fields_with_spans = a3["verified_spans"].map(lambda ss: sorted({s["source_field"] for s in ss}))
    summary = pd.DataFrame([
        ("Records", n),
        ("Entity detected (not NO_ENTITY)", pct(int(etype.ne("NO_ENTITY").sum()), n)),
        ("NO_ENTITY", pct(int(etype.eq("NO_ENTITY").sum()), n)),
        ("BANK", pct(int(etype.eq("BANK").sum()), n)),
        (f"Entity confidence > {thr} (protection applies)", pct(int(a3["protection_applies"].fillna(False).astype(bool).sum()), n)),
        (f"Entity confidence == {thr} exactly", int(conf.eq(thr).sum())),
        ("Records with >= 1 verified span", pct(int(n_verified.gt(0).sum()), n)),
        ("Returned spans verified / returned", f"{int(n_verified.sum())} / {int(n_verified.sum() + n_rejected.sum())}"),
        ("Records with a rejected (hallucinated/invalid) span", int(n_rejected.gt(0).sum())),
        ("Entities spanning more than one source field", int(fields_with_spans.map(len).gt(1).sum())),
    ], columns=["metric", "value"])
    display(summary)
    print("entity type distribution:"); display(etype.value_counts().rename_axis("entity_type").to_frame("rows"))
    print("source fields carrying verified spans:")
    display(pd.Series([f for fs in fields_with_spans for f in fs]).value_counts().rename_axis("source_field").to_frame("records"))
    print("entity confidence distribution:"); display(conf.describe().round(3).to_frame().T)
    rej = a3.loc[n_rejected.gt(0), [RECORD_ID, "rejected_spans"]]
    if len(rej): print("rejected spans (never protect anything):"); show(rej, n=10)

## 5 — Side-by-side master table

One row per `RECORD_ID`, shared fields once, experiment columns prefixed `EXP01_` / `EXP02_` / `EXP03_`.

In [ ]:
def compare_experiments(runs: dict, order: list[str]) -> pd.DataFrame:
    base = runs[order[0]]["frame"]
    m = base[[RECORD_ID, *SOURCE_FIELDS, C["predicted_town"], C["predicted_country"], C["predicted_town_probability"],
              C["predicted_country_probability"], C["combined_address_cleaned"]]].rename(columns={
        C["predicted_town"]: "predicted_town", C["predicted_country"]: "predicted_country",
        C["predicted_town_probability"]: "town_probability", C["predicted_country_probability"]: "country_probability",
        C["combined_address_cleaned"]: "combined_address_cleaned"})
    for exp_id in order:
        tag = TAG[exp_id]; run = runs[exp_id]
        f = run["frame"][[RECORD_ID, C["combined_address_retracted"], C["combined_address_retracted_comments"], C["hitl_state"]]].rename(
            columns={C["combined_address_retracted"]: f"{tag}_retracted_address", C["combined_address_retracted_comments"]: f"{tag}_comment",
                     C["hitl_state"]: f"{tag}_HITL_state"})
        m = m.merge(f, on=RECORD_ID, how="outer")
        a = run["audit"].rename(columns=lambda c: c if c == RECORD_ID else f"{tag}_{c}")
        m = m.merge(a, on=RECORD_ID, how="left")
    e = runs["EXP-03"]["frame"][[RECORD_ID, C["entity_name"], C["entity_type"], C["entity_rationale"]]].rename(
        columns={C["entity_name"]: "entity_name", C["entity_type"]: "entity_type", C["entity_rationale"]: "entity_rationale"})
    m = m.merge(e, on=RECORD_ID, how="left")
    m["town_probability"] = as_num(m["town_probability"]); m["country_probability"] = as_num(m["country_probability"])
    m["exp02_vs_exp03_changed"] = m["EXP02_retracted_address"].ne(m["EXP03_retracted_address"])
    m["exp01_vs_exp03_changed"] = m["EXP01_retracted_address"].ne(m["EXP03_retracted_address"])
    m["exp01_vs_exp02_changed"] = m["EXP01_retracted_address"].ne(m["EXP02_retracted_address"])
    return m


if ready() and not CONTAMINATED:
    master = compare_experiments(RUNS, ORDER)
    MASTER_VIEW = [RECORD_ID, *SOURCE_FIELDS, "predicted_town", "predicted_country", "town_probability", "country_probability",
                   "entity_name", "entity_type", "EXP01_retracted_address", "EXP02_retracted_address", "EXP03_retracted_address",
                   "exp02_vs_exp03_changed"]
    print(f"{len(master)} records; EXP-03 differs from EXP-02 in {int(master['exp02_vs_exp03_changed'].sum())}, from EXP-01 in {int(master['exp01_vs_exp03_changed'].sum())}")
    show(master[MASTER_VIEW], n=10)

## 6 — Three-way retraction comparison

HITL and AUTO_ACCEPT must be identical across all three. `Line 1 changed` is allowed to be non-zero
for EXP-03; `Entity-span altered` must be 0.

In [ ]:
from models.swft_tc.src.retraction import token_phrase_matches

ORGANISATION_TOKENS = ("BANK", "BANQUE", "BANCO", "CITIBANK", "CORP", "CORPORATION", "INC", "LIMITED", "LTD", "PLC", "BRANCH",
                       "OFFICE", "TRUST", "FINANCIAL", "SECURITIES", "CAPITAL", "GROUP", "AUTHORITY", "COMPANY", "SA", "SAE", "LLC", "NA")


def entity_collision_candidate(frame: pd.DataFrame) -> pd.Series:
    """Notebook-only manual-review heuristic (from the EXP-01 notebook). Not a correctness label."""
    cleaned = frame[C["combined_address_cleaned"]].astype(str)
    has_org = cleaned.map(lambda s: any(token_phrase_matches(s, tok) for tok in ORGANISATION_TOKENS))
    verified = as_bool(frame[C["predicted_town_exists"]]) | as_bool(frame[C["predicted_country_exists"]])
    return has_org & verified & cleaned.ne(frame[C["combined_address_retracted"]].astype(str))


def summarize_retraction_metrics(run: dict, entity_spans: pd.Series | None, bank_mask: pd.Series | None) -> dict:
    f = run["frame"]; a = run["audit"].set_index(RECORD_ID).reindex(f[RECORD_ID]).reset_index()
    ents = a["retracted_entities"].astype(str)
    cleaned, retracted = f[C["combined_address_cleaned"]], f[C["combined_address_retracted"]]
    out = {"Records": len(f), "Any retraction": int(cleaned.ne(retracted).sum()), "No retraction": int(cleaned.eq(retracted).sum()),
           "Town retracted": int(ents.str.contains("town").sum()), "Country retracted": int(ents.str.contains("country").sum())}
    for i in range(1, 5):
        out[f"Line {i} changed"] = int(a[f"line{i}_changed"].fillna(False).astype(bool).sum())
    # "Entity-span altered": judged against EXP-03's verified spans for EVERY experiment, so EXP-01/EXP-02 are
    # scored on how many of the same spans they destroyed.
    if entity_spans is not None:
        detail_after = {str(d["record_id"]): d["groups"][GROUP]["retraction"]["actual_column_after_retraction"] for d in run["detail"] if GROUP in d.get("groups", {})}
        altered = [any(s["text"] not in detail_after.get(rid, {}).get(s["source_field"], "") for s in spans)
                   for rid, spans in entity_spans.items()]
        out["Entity-span altered"] = int(sum(altered))
        out["Bank entity altered"] = int(sum(alt for alt, rid in zip(altered, entity_spans.index) if bank_mask is not None and bank_mask.get(rid, False)))
    else:
        out["Entity-span altered"] = out["Bank entity altered"] = None
    out["HITL"] = int(as_bool(f[C["hitl_flag"]]).sum()); out["AUTO_ACCEPT"] = int((~as_bool(f[C["hitl_flag"]])).sum())
    out["Possible entity collision candidates"] = int(entity_collision_candidate(f).sum())
    return out


if ready() and not CONTAMINATED:
    a3 = RUNS[CHALLENGER_ID]["audit"].set_index(RECORD_ID)
    verified_spans = a3["verified_spans"].loc[lambda s: s.map(len) > 0]          # EXP-03's verified spans, by record
    bank_mask = RUNS[CHALLENGER_ID]["frame"].set_index(RECORD_ID)[C["entity_type"]].eq("BANK")
    three_way = pd.DataFrame({exp_id: summarize_retraction_metrics(RUNS[exp_id], verified_spans, bank_mask) for exp_id in ORDER})
    display(three_way)
    for metric in ("HITL", "AUTO_ACCEPT"):
        if three_way.loc[metric].nunique() != 1:
            print(f"!!! UNEXPECTED: {metric} differs across experiments — routing must not change.")
    if three_way.at["Entity-span altered", CHALLENGER_ID] != 0:
        print("!!! EXP-03 altered a verified, protected entity span — the success criterion failed.")
    else:
        print("EXP-03 success criterion: verified entity spans altered = 0.")

## 7 — BANK review

Every record EXP-03 classified as BANK, with the three retracted addresses side by side. The
high-priority cases from the brief are shown first when present.

In [ ]:
HIGH_PRIORITY_BANKS = ["2152", "1578", "1450", "1150", "1212", "1233"]

if ready() and not CONTAMINATED:
    a3 = RUNS[CHALLENGER_ID]["audit"].set_index(RECORD_ID)
    bank = master.loc[master["entity_type"].eq("BANK")].copy()
    bank["entity_confidence"] = bank[RECORD_ID].map(a3["entity_confidence"])
    bank["verified_entity_spans"] = bank[RECORD_ID].map(a3["verified_spans"].map(lambda ss: "; ".join(f"{s['source_field']}:'{s['text']}'" for s in ss)))
    bank["priority"] = bank[RECORD_ID].map(lambda r: HIGH_PRIORITY_BANKS.index(r) if r in HIGH_PRIORITY_BANKS else 99)
    bank = bank.sort_values(["priority", RECORD_ID])
    BANK_VIEW = [RECORD_ID, *SOURCE_FIELDS, "entity_name", "entity_type", "entity_rationale", "entity_confidence", "verified_entity_spans",
                 "predicted_town", "predicted_country", "EXP01_retracted_address", "EXP02_retracted_address", "EXP03_retracted_address"]
    print(f"BANK entities: {len(bank)} | high-priority present: {[r for r in HIGH_PRIORITY_BANKS if r in set(bank[RECORD_ID])]}")
    show(bank[BANK_VIEW], n=20)

## 8 — EXP-02 vs EXP-03 categories

Where the two challengers disagree, and why, from the audit fields.

In [ ]:
def exp02_vs_exp03_category(row) -> str:
    if not row["exp02_vs_exp03_changed"]:
        return "SAME_AS_EXP02"
    e3_spans_outside_l1 = any(s["source_field"] != SOURCE_FIELDS[0] for s in (row.get("EXP03_protected_entity_spans") or []))
    e3_protected = bool(row.get("EXP03_protected_entity_spans"))
    e3_line1 = bool(row.get("EXP03_line1_changed"))
    e2_line1_protected = row.get("EXP02_town_skip_reason") == "protected_source_only" or row.get("EXP02_country_skip_reason") == "protected_source_only" \
        or ("protected line" in str(row.get("EXP02_comment", "")))
    if e3_line1 and not e3_protected and e2_line1_protected:
        return "EXP03_ENTITY_DETECTION_MISSED_EXP02_PROTECTION"        # EXP-02 shielded line 1; EXP-03 found no span and cut it
    if e3_line1 and e3_protected:
        return "EXP03_RETRACTED_NON_ENTITY_TEXT_THAT_EXP02_BLANKET_PROTECTED"   # span kept, non-entity line-1 text removed
    if e3_spans_outside_l1 and (row.get("EXP03_town_protected_by_entity", 0) or row.get("EXP03_country_protected_by_entity", 0)):
        return "EXP03_PROTECTED_ENTITY_OUTSIDE_LINE1"
    if e3_protected and len(row["EXP03_retracted_address"]) > len(row["EXP02_retracted_address"]):
        return "EXP03_ENTITY_OVERPROTECTED_LOCATION_TEXT"       # EXP-03 kept more than EXP-02 did, via a span
    return "OTHER"


if ready() and not CONTAMINATED:
    master["exp02_vs_exp03_category"] = master.apply(exp02_vs_exp03_category, axis=1)
    display(master["exp02_vs_exp03_category"].value_counts().rename_axis("category").to_frame("rows"))
    for cat in [c for c in master["exp02_vs_exp03_category"].unique() if c != "SAME_AS_EXP02"]:
        print(f"\n{cat}")
        show(master.loc[master["exp02_vs_exp03_category"].eq(cat), [RECORD_ID, "address_line_1", "entity_name", "entity_type",
                        "EXP02_retracted_address", "EXP03_retracted_address"]], n=8)

## 9 — Improvement over EXP-01

Entity spans (as EXP-03 verified them) that EXP-01 destroyed versus EXP-03 preserved, and the same
for BANK names.

In [ ]:
if ready() and not CONTAMINATED:
    r = three_way
    print(f"- entity spans destroyed by EXP-01 : {r.at['Entity-span altered', 'EXP-01']} of {len(verified_spans)} verified spans")
    print(f"- entity spans destroyed by EXP-02 : {r.at['Entity-span altered', 'EXP-02']}")
    print(f"- entity spans preserved by EXP-03 : {len(verified_spans) - r.at['Entity-span altered', 'EXP-03']} (altered {r.at['Entity-span altered', 'EXP-03']})")
    print(f"- bank names damaged by EXP-01     : {r.at['Bank entity altered', 'EXP-01']} of {int(bank_mask.sum())} BANK entities")
    print(f"- bank names preserved by EXP-03   : {int(bank_mask.sum()) - r.at['Bank entity altered', 'EXP-03']}")

## 10 — Manual ground-truth review set

Changed and high-risk cases, with review flags. `review_label` is left empty for a human
(`EXPECTED_PRESERVE` / `EXPECTED_RETRACT` / `UNCERTAIN`); nothing here labels EXP-03 as correct.

In [ ]:
if ready() and not CONTAMINATED:
    a3 = RUNS[CHALLENGER_ID]["audit"].set_index(RECORD_ID)
    thr = EXPERIMENTS[CHALLENGER_ID]["entity"].protection_confidence_threshold or 0.0
    conf = as_num(master[RECORD_ID].map(a3["entity_confidence"]))
    spans = master[RECORD_ID].map(a3["verified_spans"])
    name_has_geo = [bool(n) and n != "NO_ENTITY" and (bool(token_phrase_matches(n, t)) or bool(token_phrase_matches(n, c)))
                    for n, t, c in zip(master["entity_name"].astype(str), master["predicted_town"].astype(str), master["predicted_country"].astype(str))]
    flags = {
        "exp01_vs_exp02_changed": master["exp01_vs_exp02_changed"],
        "entity_span_outside_line1": spans.map(lambda ss: any(s["source_field"] != SOURCE_FIELDS[0] for s in ss)),
        "entity_confidence_near_threshold": conf.between(thr - 0.05, thr + 0.05),
        "exp03_changed_what_exp02_preserved": master["exp02_vs_exp03_changed"] & master["EXP03_retracted_address"].str.len().lt(master["EXP02_retracted_address"].str.len()),
        "exp03_failed_to_protect_a_bank": master["entity_type"].eq("BANK") & master[RECORD_ID].map(a3["span_altered"]).fillna(False).astype(bool),
        "returned_span_rejected": master[RECORD_ID].map(a3["rejected_spans"].map(len)).gt(0),
        "entity_type_bank": master["entity_type"].eq("BANK"),
        "entity_name_contains_predicted_town_or_country": pd.Series(name_has_geo, index=master.index),
    }
    master["review_flags"] = [",".join(k for k, v in flags.items() if bool(v.iloc[i])) for i in range(len(master))]
    review = master.loc[master["review_flags"].ne("")].copy()
    review["entity_confidence"] = conf.loc[review.index]
    review["review_label"] = ""
    REVIEW_VIEW = [RECORD_ID, *SOURCE_FIELDS, "predicted_town", "predicted_country", "entity_name", "entity_type", "entity_confidence",
                   "EXP01_retracted_address", "EXP02_retracted_address", "EXP03_retracted_address", "exp02_vs_exp03_category", "review_flags", "review_label"]
    print("review flags (a record may carry several):"); display(pd.DataFrame({k: [int(v.sum())] for k, v in flags.items()}, index=["rows"]).T)
    show(review[REVIEW_VIEW], n=15)

## 11 — Conclusions

Generated from the values above. Policy behaviour only.

In [ ]:
if ready() and not CONTAMINATED:
    n = len(master); r = three_way
    print(f"- Integrity: 0 of {n} records contaminated across EXP-01/EXP-02/EXP-03; the comparison is controlled.")
    print(f"- EXP-03 detected an entity in {int(master['entity_type'].ne('NO_ENTITY').sum())} records ({int(bank_mask.sum())} BANK) and "
          f"protected verified spans in {len(verified_spans)}; it altered {r.at['Entity-span altered', 'EXP-03']} of them.")
    print(f"- EXP-01 destroyed {r.at['Entity-span altered', 'EXP-01']} of those spans and EXP-02 {r.at['Entity-span altered', 'EXP-02']}.")
    print(f"- EXP-03 changed line 1 in {r.at['Line 1 changed', 'EXP-03']} records (allowed where the text is outside a verified span); "
          f"EXP-02 in {r.at['Line 1 changed', 'EXP-02']}, EXP-01 in {r.at['Line 1 changed', 'EXP-01']}.")
    print(f"- The retracted address differs between EXP-02 and EXP-03 in {int(master['exp02_vs_exp03_changed'].sum())} records; categories in section 8.")
    print(f"- HITL and AUTO_ACCEPT are identical across the three experiments: {r.loc['HITL'].nunique() == 1 and r.loc['AUTO_ACCEPT'].nunique() == 1}.")
    print("- These results measure policy behaviour; they do not establish semantic correctness without labelled retraction ground truth.")

## 12 — Optional exports

Off by default. `EXPORT_COMPARISON` writes the three-way comparison; `EXPORT_REVIEW` writes the manual
review set with an empty `review_label` column. Both are git-ignored (raw addresses).

In [ ]:
EXPORT_COMPARISON = False
EXPORT_REVIEW = False

if ready() and not CONTAMINATED and EXPORT_COMPARISON:
    path = OUTPUT_ROOT / "exp03_vs_exp02_vs_exp01_retraction_comparison.csv"
    master[[RECORD_ID, *SOURCE_FIELDS, "predicted_town", "predicted_country", "town_probability", "country_probability", "entity_name",
            "entity_type", "EXP01_retracted_address", "EXP02_retracted_address", "EXP03_retracted_address", "exp02_vs_exp03_changed",
            "exp02_vs_exp03_category"]].to_csv(path, index=False, encoding="utf-8")
    print(f"wrote {len(master)} rows -> {_rel(path)}")
else:
    print("Comparison export disabled.")
if ready() and not CONTAMINATED and EXPORT_REVIEW:
    path = OUTPUT_ROOT / "exp03_manual_review.csv"
    review[REVIEW_VIEW].to_csv(path, index=False, encoding="utf-8"); print(f"wrote {len(review)} rows -> {_rel(path)}")
else:
    print("Review export disabled.")